# Cálculo Numérico — Lista de Exercícios
**Sistemas lineares e implementação numérica** · UNIFESP

Resolução completa da lista: resolução manual (Partes 1 a 2 e item 5a), implementação em Python (Parte 3 e item 5b) e respostas conceituais, tudo neste notebook.

| Parte | Exercícios |
|---|---|
| 1. Conceitos e resolução manual | 1, 2 |
| 2. Propriedades e interpretação | 3 |
| 3. Implementação numérica | 4 |
| 4. Aplicação | 5 |

**Como executar no Google Colab:** menu *Ambiente de execução → Executar tudo* (ou `Ctrl+F9`). Só são usados `numpy` e `fractions`, que já vêm instalados no Colab.

In [1]:
import numpy as np
from fractions import Fraction as Fr   # frações exatas, para mostrar a resolução manual sem erro de arredondamento

np.set_printoptions(precision=6, suppress=True)

---
## Parte 1 — Conceitos e resolução manual

### Exercício 1 — Sistemas lineares na forma matricial

Um sistema linear com $n$ equações e $n$ incógnitas se escreve como $A\mathbf{x}=\mathbf{b}$, em que $A$ é a matriz dos coeficientes, $\mathbf{x}$ é o vetor de incógnitas e $\mathbf{b}$ é o vetor independente (lado direito das equações).

**a)**
$$
\underbrace{\begin{bmatrix} 2 & 1 \\ 1 & -3 \end{bmatrix}}_{A}
\underbrace{\begin{bmatrix} x \\ y \end{bmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{bmatrix} 7 \\ -5 \end{bmatrix}}_{\mathbf{b}}
$$

**b)**
$$
\underbrace{\begin{bmatrix} 1 & 2 & -1 \\ 3 & -1 & 2 \\ 2 & 1 & 1 \end{bmatrix}}_{A}
\underbrace{\begin{bmatrix} x \\ y \\ z \end{bmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{bmatrix} 2 \\ 9 \\ 7 \end{bmatrix}}_{\mathbf{b}}
$$

A célula abaixo monta essas matrizes no NumPy e confere a solução de cada sistema (só como verificação; o exercício pede apenas a escrita matricial).

In [2]:
# Exercício 1: montando A e b com NumPy
A_a = np.array([[2,  1],
                [1, -3]], dtype=float)
b_a = np.array([7, -5], dtype=float)

A_b = np.array([[1,  2, -1],
                [3, -1,  2],
                [2,  1,  1]], dtype=float)
b_b = np.array([2, 9, 7], dtype=float)

# Conferência da solução de cada sistema
print("a) (x, y)    =", np.linalg.solve(A_a, b_a))   # (16/7, 17/7)
print("b) (x, y, z) =", np.linalg.solve(A_b, b_b))   # (2, 1, 2)

a) (x, y)    = [2.285714 2.428571]
b) (x, y, z) = [2. 1. 2.]


### Exercício 2 — Método da eliminação de Gauss

$$
\begin{cases}
2x + y - z = 8\\
-3x - y + 2z = -11\\
-2x + y + 2z = -3
\end{cases}
$$

**Matriz aumentada inicial**
$$
\left[\begin{array}{ccc|c}
2 & 1 & -1 & 8\\
-3 & -1 & 2 & -11\\
-2 & 1 & 2 & -3
\end{array}\right]
$$

**Etapa 1 — pivô $a_{11}=2$.** Multiplicadores: $m_{21}=\dfrac{-3}{2}=-\dfrac32$ e $m_{31}=\dfrac{-2}{2}=-1$.

$R_2 \leftarrow R_2 + \tfrac32 R_1$ e $R_3 \leftarrow R_3 + R_1$:
$$
\left[\begin{array}{ccc|c}
2 & 1 & -1 & 8\\
0 & \tfrac12 & \tfrac12 & 1\\
0 & 2 & 1 & 5
\end{array}\right]
$$

**Etapa 2 — pivô $a_{22}=\tfrac12$.** Multiplicador: $m_{32}=\dfrac{2}{1/2}=4$.

$R_3 \leftarrow R_3 - 4R_2$:
$$
\left[\begin{array}{ccc|c}
2 & 1 & -1 & 8\\
0 & \tfrac12 & \tfrac12 & 1\\
0 & 0 & -1 & 1
\end{array}\right]
$$

**Substituição regressiva**

- Linha 3: $-z = 1 \Rightarrow z = -1$
- Linha 2: $\tfrac12 y + \tfrac12(-1) = 1 \Rightarrow \tfrac12 y = \tfrac32 \Rightarrow y = 3$
- Linha 1: $2x + 3 - (-1) = 8 \Rightarrow 2x = 4 \Rightarrow x = 2$

$$\boxed{(x, y, z) = (2,\ 3,\ -1)}$$

A célula abaixo repete as mesmas operações com frações exatas para conferir cada matriz.

In [3]:
def mostra(M, titulo):
    """Imprime a matriz aumentada [A | b] com frações exatas."""
    print(titulo)
    for linha in M:
        coef = "  ".join(f"{str(v):>5}" for v in linha[:-1])
        print(f"  [ {coef}  | {str(linha[-1]):>5} ]")
    print()

# Matriz aumentada [A | b] com frações exatas
M = [[Fr(2),  Fr(1), Fr(-1),  Fr(8)],
     [Fr(-3), Fr(-1), Fr(2), Fr(-11)],
     [Fr(-2), Fr(1),  Fr(2),  Fr(-3)]]
mostra(M, "Matriz aumentada inicial")

# Etapa 1: zerar a coluna 1 abaixo do pivô (a11 = 2)
m21 = M[1][0] / M[0][0]                       # -3/2
M[1] = [a - m21 * b for a, b in zip(M[1], M[0])]
m31 = M[2][0] / M[0][0]                       # -1
M[2] = [a - m31 * b for a, b in zip(M[2], M[0])]
mostra(M, f"Etapa 1: R2 <- R2 - ({m21})R1  e  R3 <- R3 - ({m31})R1")

# Etapa 2: zerar a coluna 2 abaixo do pivô (a22 = 1/2)
m32 = M[2][1] / M[1][1]                       # 4
M[2] = [a - m32 * b for a, b in zip(M[2], M[1])]
mostra(M, f"Etapa 2: R3 <- R3 - ({m32})R2  (matriz triangular superior)")

# Substituição regressiva (de baixo para cima)
z = M[2][3] / M[2][2]
y = (M[1][3] - M[1][2] * z) / M[1][1]
x = (M[0][3] - M[0][1] * y - M[0][2] * z) / M[0][0]
print(f"Solução: x = {x}, y = {y}, z = {z}")

Matriz aumentada inicial
  [     2      1     -1  |     8 ]
  [    -3     -1      2  |   -11 ]
  [    -2      1      2  |    -3 ]

Etapa 1: R2 <- R2 - (-3/2)R1  e  R3 <- R3 - (-1)R1
  [     2      1     -1  |     8 ]
  [     0    1/2    1/2  |     1 ]
  [     0      2      1  |     5 ]

Etapa 2: R3 <- R3 - (4)R2  (matriz triangular superior)
  [     2      1     -1  |     8 ]
  [     0    1/2    1/2  |     1 ]
  [     0      0     -1  |     1 ]

Solução: x = 2, y = 3, z = -1


---
## Parte 2 — Propriedades e interpretação

### Exercício 3 — Número de soluções

Regra usada: reduz-se a matriz aumentada $[A\,|\,\mathbf{b}]$ e comparam-se o posto de $A$, o posto de $[A\,|\,\mathbf{b}]$ e o número de incógnitas $n$.

- posto$(A)$ = posto$([A|\mathbf{b}])$ = $n$: **solução única**;
- posto$(A)$ = posto$([A|\mathbf{b}])$ < $n$: **infinitas soluções**;
- posto$(A)$ < posto$([A|\mathbf{b}])$: **nenhuma solução** (aparece uma linha $0=c$ com $c\neq0$).

**a)** $x+y=3,\ 2x+2y=6$

$$
\left[\begin{array}{cc|c} 1 & 1 & 3\\ 2 & 2 & 6\end{array}\right]
\xrightarrow{R_2\leftarrow R_2-2R_1}
\left[\begin{array}{cc|c} 1 & 1 & 3\\ 0 & 0 & 0\end{array}\right]
$$

A segunda linha vira $0=0$. Posto$(A)$ = posto$([A|\mathbf{b}])$ = 1 < 2, então há **infinitas soluções**: $x = 3 - t,\ y = t$, com $t\in\mathbb{R}$. Geometricamente, as duas equações descrevem a mesma reta.

**b)** $x+y=3,\ 2x+2y=7$

$$
\left[\begin{array}{cc|c} 1 & 1 & 3\\ 2 & 2 & 7\end{array}\right]
\xrightarrow{R_2\leftarrow R_2-2R_1}
\left[\begin{array}{cc|c} 1 & 1 & 3\\ 0 & 0 & 1\end{array}\right]
$$

A segunda linha vira $0=1$, que é impossível. Posto$(A)$ = 1 < posto$([A|\mathbf{b}])$ = 2, então **não há solução**. Geometricamente, as retas são paralelas e distintas.

**c)** $x+y=3,\ 2x-y=0$

$$
\left[\begin{array}{cc|c} 1 & 1 & 3\\ 2 & -1 & 0\end{array}\right]
\xrightarrow{R_2\leftarrow R_2-2R_1}
\left[\begin{array}{cc|c} 1 & 1 & 3\\ 0 & -3 & -6\end{array}\right]
$$

Da linha 2, $y = 2$; da linha 1, $x = 3-2 = 1$. Posto$(A)$ = posto$([A|\mathbf{b}])$ = 2 = $n$, então a **solução é única**: $(x,y)=(1,2)$. Geometricamente, as retas se cruzam em um único ponto.

A célula abaixo confirma a classificação pelos postos.

In [4]:
# Exercício 3: classificação pelos postos de A e de [A | b]
casos = {
    "a) x+y=3, 2x+2y=6": ([[1, 1], [2,  2]], [3, 6]),
    "b) x+y=3, 2x+2y=7": ([[1, 1], [2,  2]], [3, 7]),
    "c) x+y=3, 2x-y=0 ": ([[1, 1], [2, -1]], [3, 0]),
}

for nome, (A, b) in casos.items():
    A = np.array(A, dtype=float)
    Ab = np.column_stack([A, b])                       # matriz aumentada [A | b]
    rA, rAb, n = np.linalg.matrix_rank(A), np.linalg.matrix_rank(Ab), A.shape[1]

    if rA < rAb:
        tipo = "nenhuma solução"
    elif rA == n:
        tipo = "solução única"
    else:
        tipo = "infinitas soluções"

    print(f"{nome}: posto(A)={rA}, posto([A|b])={rAb}, n={n}  ->  {tipo}")

a) x+y=3, 2x+2y=6: posto(A)=1, posto([A|b])=1, n=2  ->  infinitas soluções
b) x+y=3, 2x+2y=7: posto(A)=1, posto([A|b])=2, n=2  ->  nenhuma solução
c) x+y=3, 2x-y=0 : posto(A)=2, posto([A|b])=2, n=2  ->  solução única


---
## Parte 3 — Implementação numérica em Python

### Exercício 4 — Eliminação de Gauss sem pivoteamento

A função abaixo recebe `A` e `b`, forma a matriz aumentada, elimina os elementos abaixo dos pivôs, aplica a substituição regressiva e devolve a solução.

In [5]:
def eliminacao_gauss(A, b, tol=1e-12):
    """Resolve A x = b por eliminação de Gauss SEM pivoteamento.

    Parâmetros
    ----------
    A : matriz quadrada n x n (lista de listas ou array)
    b : vetor independente de tamanho n
    tol : abaixo desse valor um pivô é considerado nulo

    Retorna
    -------
    x : array com a solução
    """
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1)
    n = A.shape[0]

    # 1) Matriz aumentada [A | b]
    Ab = np.column_stack([A, b])

    # 2) Eliminação: zera os elementos abaixo de cada pivô
    for k in range(n - 1):
        if abs(Ab[k, k]) < tol:
            raise ZeroDivisionError(
                f"Pivô nulo na posição ({k+1},{k+1}): sem pivoteamento o método falha.")
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]          # multiplicador m_ik
            Ab[i, k:] -= m * Ab[k, k:]       # Ri <- Ri - m * Rk

    if abs(Ab[n - 1, n - 1]) < tol:
        raise ZeroDivisionError("Último pivô nulo: sistema singular.")

    # 3) Substituição regressiva (da última linha para a primeira)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, n] - Ab[i, i + 1:n] @ x[i + 1:n]) / Ab[i, i]

    return x

In [6]:
# Teste com o sistema do exercício 2 e comparação com numpy.linalg.solve
A2 = [[ 2,  1, -1],
      [-3, -1,  2],
      [-2,  1,  2]]
b2 = [8, -11, -3]

x_gauss = eliminacao_gauss(A2, b2)
x_numpy = np.linalg.solve(A2, b2)

print("Gauss (nossa função):", x_gauss)
print("numpy.linalg.solve:  ", x_numpy)
print("Diferença máxima:    ", np.max(np.abs(x_gauss - x_numpy)))
print("Resíduo ||Ax - b||:  ", np.linalg.norm(np.array(A2) @ x_gauss - b2))

Gauss (nossa função): [ 2.  3. -1.]
numpy.linalg.solve:   [ 2.  3. -1.]
Diferença máxima:     1.2212453270876722e-15
Resíduo ||Ax - b||:   0.0


In [7]:
# Limitação do método sem pivoteamento: pivô nulo
try:
    eliminacao_gauss([[0, 2],
                      [1, 1]], [4, 3])      # solução existe: (1, 2)
except ZeroDivisionError as erro:
    print("Erro esperado:", erro)

Erro esperado: Pivô nulo na posição (1,1): sem pivoteamento o método falha.


**Interpretação.** A solução obtida, $(2,\,3,\,-1)$, coincide com a resolução manual do exercício 2 e com `numpy.linalg.solve`. A diferença entre os dois métodos é da ordem do erro de arredondamento da aritmética de ponto flutuante (aqui, cerca de $10^{-15}$), e o resíduo $\|A\mathbf{x}-\mathbf{b}\|$ também é desprezível.

Neste sistema os pivôs ($2$, $\tfrac12$ e $-1$) são todos não nulos, por isso o pivoteamento não foi necessário. A última célula mostra a limitação do método: o sistema $0x+2y=4,\ x+y=3$ tem solução, mas o pivô $a_{11}=0$ impede a eliminação sem trocar linhas. Por isso, na prática, usa-se o **pivoteamento parcial**, que também reduz a propagação de erros quando um pivô é muito pequeno.

---
## Parte 4 — Aplicação

### Exercício 5 — Modelo de balanço térmico

$$
\begin{bmatrix} 4 & -1 & 0\\ -1 & 4 & -1\\ 0 & -1 & 3\end{bmatrix}
\begin{bmatrix} T_1\\ T_2\\ T_3\end{bmatrix}
=
\begin{bmatrix} 15\\ 10\\ 10\end{bmatrix}
$$

**a) Resolução manual**

Matriz aumentada inicial:
$$
\left[\begin{array}{ccc|c}
4 & -1 & 0 & 15\\
-1 & 4 & -1 & 10\\
0 & -1 & 3 & 10
\end{array}\right]
$$

**Etapa 1 — pivô $a_{11}=4$.** $m_{21}=\dfrac{-1}{4}$ e $m_{31}=0$. Então $R_2 \leftarrow R_2 + \tfrac14 R_1$:
$$
\left[\begin{array}{ccc|c}
4 & -1 & 0 & 15\\
0 & \tfrac{15}{4} & -1 & \tfrac{55}{4}\\
0 & -1 & 3 & 10
\end{array}\right]
$$

**Etapa 2 — pivô $a_{22}=\tfrac{15}{4}$.** $m_{32}=\dfrac{-1}{15/4}=-\dfrac{4}{15}$. Então $R_3 \leftarrow R_3 + \tfrac{4}{15}R_2$:
$$
\left[\begin{array}{ccc|c}
4 & -1 & 0 & 15\\
0 & \tfrac{15}{4} & -1 & \tfrac{55}{4}\\
0 & 0 & \tfrac{41}{15} & \tfrac{41}{3}
\end{array}\right]
$$

pois $3-\tfrac{4}{15}=\tfrac{41}{15}$ e $10+\tfrac{4}{15}\cdot\tfrac{55}{4}=10+\tfrac{11}{3}=\tfrac{41}{3}$.

**Substituição regressiva**

- $\tfrac{41}{15}T_3=\tfrac{41}{3} \Rightarrow T_3 = 5$
- $\tfrac{15}{4}T_2 - 5 = \tfrac{55}{4} \Rightarrow \tfrac{15}{4}T_2=\tfrac{75}{4} \Rightarrow T_2 = 5$
- $4T_1 - 5 = 15 \Rightarrow T_1 = 5$

$$\boxed{T_1 = T_2 = T_3 = 5}$$

**Verificação:** $4(5)-5=15$ ✓, $-5+4(5)-5=10$ ✓, $-5+3(5)=10$ ✓.

**b) Implementação em Python**

In [8]:
# Exercício 5b: balanço térmico resolvido com a função do exercício 4
A5 = [[ 4, -1,  0],
      [-1,  4, -1],
      [ 0, -1,  3]]
b5 = [15, 10, 10]

T_gauss = eliminacao_gauss(A5, b5)
T_numpy = np.linalg.solve(A5, b5)

print("T1, T2, T3 (Gauss):", T_gauss)
print("T1, T2, T3 (numpy):", T_numpy)
print("Resíduo ||AT - b||:", np.linalg.norm(np.array(A5) @ T_gauss - b5))

T1, T2, T3 (Gauss): [5. 5. 5.]
T1, T2, T3 (numpy): [5. 5. 5.]
Resíduo ||AT - b||: 0.0


**Interpretação.** As três temperaturas valem $5$, igual ao resultado manual. Cada linha do sistema é um balanço de energia em um nó: o termo da diagonal ($4$, $4$ e $3$) é maior que a soma dos acoplamentos com os vizinhos, e o lado direito representa a fonte de calor. Com $T_1=T_2=T_3=5$, cada equação se verifica exatamente.

A matriz é simétrica e diagonalmente dominante, o que garante pivôs não nulos ($4$, $\tfrac{15}{4}$ e $\tfrac{41}{15}$), então a eliminação sem pivoteamento funciona bem neste caso.